# Dataset Integration

This notebook compares and prepares the processed datasets for final model training.

Datasets:
1. BTXRD
2. Dataset 2

No augmentation is performed here.
No images are modified in this notebook.

In [1]:
from pathlib import Path
import pandas as pd

# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    r"C:\Users\DELL\Downloads\mlproject\bone-cancer-detection"
)

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"

# Dataset 1: BTXRD
BTXRD_IMAGES = PROCESSED_ROOT / "images"
BTXRD_METADATA = PROCESSED_ROOT / "BTXRD_processed_metadata.csv"

# Dataset 2
DATASET2_ROOT = PROCESSED_ROOT / "Dataset2"

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_ROOT)

print("\nBTXRD metadata exists:", BTXRD_METADATA.exists())
print("BTXRD images folder exists:", BTXRD_IMAGES.exists())

print("\nDataset 2 exists:", DATASET2_ROOT.exists())

Project root: C:\Users\DELL\Downloads\mlproject\bone-cancer-detection
Processed data: C:\Users\DELL\Downloads\mlproject\bone-cancer-detection\data\processed

BTXRD metadata exists: True
BTXRD images folder exists: True

Dataset 2 exists: True


In [2]:
# ============================================================
# LOAD METADATA
# ============================================================

# Load BTXRD metadata
btxrd = pd.read_csv(BTXRD_METADATA)

# Load Dataset 2 metadata
dataset2_parts = []

for split in ["train", "valid", "test"]:
    metadata_path = DATASET2_ROOT / split / "metadata.csv"

    df_split = pd.read_csv(metadata_path)
    df_split["split"] = split

    dataset2_parts.append(df_split)

dataset2 = pd.concat(dataset2_parts, ignore_index=True)

# ============================================================
# DISPLAY BASIC INFORMATION
# ============================================================

print("=" * 60)
print("BTXRD METADATA")
print("=" * 60)

print("Rows:", len(btxrd))
print("Columns:", list(btxrd.columns))
print("\nFirst 5 rows:")
display(btxrd.head())


print("\n" + "=" * 60)
print("DATASET 2 METADATA")
print("=" * 60)

print("Rows:", len(dataset2))
print("Columns:", list(dataset2.columns))
print("\nFirst 5 rows:")
display(dataset2.head())

BTXRD METADATA
Rows: 3746
Columns: ['image_id', 'cancer', 'image_path']

First 5 rows:


,image_id,cancer,image_path
0,IMG000001.jpeg,1,C:\Users\DELL\Downloads\mlproject\bone-cancer-...
1,IMG000002.jpeg,1,C:\Users\DELL\Downloads\mlproject\bone-cancer-...
2,IMG000003.jpeg,1,C:\Users\DELL\Downloads\mlproject\bone-cancer-...
3,IMG000004.jpeg,1,C:\Users\DELL\Downloads\mlproject\bone-cancer-...
4,IMG000005.jpeg,1,C:\Users\DELL\Downloads\mlproject\bone-cancer-...



DATASET 2 METADATA
Rows: 8810
Columns: ['processed_filename', 'cancer_label', 'split']

First 5 rows:


,processed_filename,cancer_label,split
0,-53-_jpg.rf.08d303d0b43c7b71c3581ec02d7390c3.jpg,0,train
1,image-no531-normal-_png.rf.08eeb57b24a9062d798...,0,train
2,IMG0000240_jpg.rf.094924a72ca1ab5e26477e50abac...,0,train
3,bone-cancer_train_2_821_png.rf.094c60132af2cb3...,1,train
4,image-no52-normal-_png.rf.08f33a992911794d2381...,0,train


## 1. Standardize Metadata

Both datasets contain the same binary target:

Cancer = 1  
No Cancer / Normal = 0

The metadata is standardized to use:
- `filename`
- `cancer`
- `dataset`
- `split`

In [3]:
# ============================================================
# STANDARDIZE BTXRD METADATA
# ============================================================

btxrd_standard = btxrd[
    ["image_id", "cancer"]
].copy()

btxrd_standard = btxrd_standard.rename(
    columns={
        "image_id": "filename"
    }
)

btxrd_standard["dataset"] = "BTXRD"

# BTXRD currently does not have train/valid/test splits
btxrd_standard["split"] = "not_split"


# ============================================================
# STANDARDIZE DATASET 2 METADATA
# ============================================================

dataset2_standard = dataset2[
    ["processed_filename", "cancer_label", "split"]
].copy()

dataset2_standard = dataset2_standard.rename(
    columns={
        "processed_filename": "filename",
        "cancer_label": "cancer"
    }
)

dataset2_standard["dataset"] = "Dataset2"


# ============================================================
# CHECK RESULTS
# ============================================================

print("BTXRD standardized:")
display(btxrd_standard.head())

print("\nDataset 2 standardized:")
display(dataset2_standard.head())

print("\nBTXRD columns:", list(btxrd_standard.columns))
print("Dataset 2 columns:", list(dataset2_standard.columns))

BTXRD standardized:


,filename,cancer,dataset,split
0,IMG000001.jpeg,1,BTXRD,not_split
1,IMG000002.jpeg,1,BTXRD,not_split
2,IMG000003.jpeg,1,BTXRD,not_split
3,IMG000004.jpeg,1,BTXRD,not_split
4,IMG000005.jpeg,1,BTXRD,not_split



Dataset 2 standardized:


,filename,cancer,split,dataset
0,-53-_jpg.rf.08d303d0b43c7b71c3581ec02d7390c3.jpg,0,train,Dataset2
1,image-no531-normal-_png.rf.08eeb57b24a9062d798...,0,train,Dataset2
2,IMG0000240_jpg.rf.094924a72ca1ab5e26477e50abac...,0,train,Dataset2
3,bone-cancer_train_2_821_png.rf.094c60132af2cb3...,1,train,Dataset2
4,image-no52-normal-_png.rf.08f33a992911794d2381...,0,train,Dataset2



BTXRD columns: ['filename', 'cancer', 'dataset', 'split']
Dataset 2 columns: ['filename', 'cancer', 'split', 'dataset']


## 2. Split BTXRD

BTXRD does not currently have train, validation, and test splits.

We create stratified splits so that the Cancer/No Cancer class distribution is maintained across all three sets.

The original BTXRD data remains unchanged.

In [5]:
# ============================================================
# CREATE STRATIFIED BTXRD SPLITS
# ============================================================

from sklearn.model_selection import train_test_split

# First split:
# 80% training, 20% temporary
btxrd_train, btxrd_temp = train_test_split(
    btxrd_standard,
    test_size=0.20,
    stratify=btxrd_standard["cancer"],
    random_state=42
)

# Second split:
# Divide the temporary 20% equally into validation and test
btxrd_valid, btxrd_test = train_test_split(
    btxrd_temp,
    test_size=0.50,
    stratify=btxrd_temp["cancer"],
    random_state=42
)

# Add split labels
btxrd_train = btxrd_train.copy()
btxrd_valid = btxrd_valid.copy()
btxrd_test = btxrd_test.copy()

btxrd_train["split"] = "train"
btxrd_valid["split"] = "valid"
btxrd_test["split"] = "test"

# Display sizes
print("BTXRD split sizes:")
print("Train:", len(btxrd_train))
print("Valid:", len(btxrd_valid))
print("Test :", len(btxrd_test))

# Display class distributions
print("\nClass distribution:")

for name, split_df in [
    ("TRAIN", btxrd_train),
    ("VALID", btxrd_valid),
    ("TEST", btxrd_test)
]:
    print(f"\n{name}")
    print(split_df["cancer"].value_counts().sort_index())

BTXRD split sizes:
Train: 2996
Valid: 375
Test : 375

Class distribution:

TRAIN
cancer
0    2722
1     274
Name: count, dtype: int64

VALID
cancer
0    341
1     34
Name: count, dtype: int64

TEST
cancer
0    341
1     34
Name: count, dtype: int64


## 3. Compare Dataset Splits

Compare the class distributions of BTXRD and Dataset 2 after creating the BTXRD splits.

This helps us understand the balance of the combined dataset before model training.

In [6]:
# ============================================================
# COMPARE SPLITS AND CLASS DISTRIBUTIONS
# ============================================================

# Combine the BTXRD splits
btxrd_split = pd.concat(
    [btxrd_train, btxrd_valid, btxrd_test],
    ignore_index=True
)

print("=" * 60)
print("BTXRD")
print("=" * 60)

print(
    pd.crosstab(
        btxrd_split["split"],
        btxrd_split["cancer"]
    )
)

print("\n" + "=" * 60)
print("DATASET 2")
print("=" * 60)

print(
    pd.crosstab(
        dataset2_standard["split"],
        dataset2_standard["cancer"]
    )
)

BTXRD
cancer     0    1
split            
test     341   34
train   2722  274
valid    341   34

DATASET 2
cancer     0     1
split             
test     488   384
train   3975  3081
valid    484   398


## 6. Prepare Standardized Metadata for Integration

Create a consistent metadata table for BTXRD and Dataset 2.

Both datasets use the same binary target:

Cancer = 1  
No Cancer / Normal = 0

The original train, validation, and test splits are preserved.
No images are moved, deleted, or augmented at this stage.

In [9]:
# ============================================================
# PREPARE STANDARDIZED METADATA
# ============================================================

# BTXRD
btxrd_final = btxrd_split[
    ["filename", "cancer", "dataset", "split"]
].copy()

# Dataset 2
dataset2_final = dataset2_standard[
    ["filename", "cancer", "dataset", "split"]
].copy()

# Make sure both datasets have the same column order
columns = ["filename", "cancer", "dataset", "split"]

btxrd_final = btxrd_final[columns]
dataset2_final = dataset2_final[columns]

# Combine metadata only
combined_metadata = pd.concat(
    [btxrd_final, dataset2_final],
    ignore_index=True
)

# ============================================================
# CHECK RESULT
# ============================================================

print("=" * 60)
print("COMBINED METADATA")
print("=" * 60)

print("Total records:", len(combined_metadata))
print("Columns:", list(combined_metadata.columns))

print("\nDataset distribution:")
print(combined_metadata["dataset"].value_counts())

print("\nSplit distribution:")
print(
    pd.crosstab(
        combined_metadata["dataset"],
        combined_metadata["split"]
    )
)

print("\nClass distribution:")
print(
    pd.crosstab(
        combined_metadata["dataset"],
        combined_metadata["cancer"]
    )
)

print("\nFirst 5 rows:")
display(combined_metadata.head())

COMBINED METADATA
Total records: 12556
Columns: ['filename', 'cancer', 'dataset', 'split']

Dataset distribution:
dataset
Dataset2    8810
BTXRD       3746
Name: count, dtype: int64

Split distribution:
split     test  train  valid
dataset                     
BTXRD      375   2996    375
Dataset2   872   7056    882

Class distribution:
cancer       0     1
dataset             
BTXRD     3404   342
Dataset2  4947  3863

First 5 rows:


,filename,cancer,dataset,split
0,IMG003215.jpeg,0,BTXRD,train
1,IMG000002.jpeg,1,BTXRD,train
2,IMG001334.jpeg,0,BTXRD,train
3,IMG001917.jpeg,0,BTXRD,train
4,IMG003088.jpeg,0,BTXRD,train


## 7. Combined Dataset Distribution

Calculate the final number of Cancer and Normal images in each combined train, validation, and test split.

This determines the class balance before augmentation and model training.

In [10]:
# ============================================================
# FINAL COMBINED CLASS DISTRIBUTION
# ============================================================

print("=" * 60)
print("FINAL COMBINED DATASET")
print("=" * 60)

combined_distribution = pd.crosstab(
    combined_metadata["split"],
    combined_metadata["cancer"]
)

# Rename columns for clarity
combined_distribution = combined_distribution.rename(
    columns={
        0: "Normal",
        1: "Cancer"
    }
)

print(combined_distribution)

print("\nTotal images per split:")
print(combined_metadata["split"].value_counts().sort_index())

print("\nTotal images:")
print(len(combined_metadata))

FINAL COMBINED DATASET
cancer  Normal  Cancer
split                 
test       829     418
train     6697    3355
valid      825     432

Total images per split:
split
test      1247
train    10052
valid     1257
Name: count, dtype: int64

Total images:
12556


## 8. Create Combined Dataset

Create a single integrated dataset by combining the corresponding train, validation, and test splits from BTXRD and Dataset 2.

Images are copied rather than moved, so the original processed datasets remain unchanged.

Dataset prefixes are added to filenames to prevent filename collisions.

In [13]:
# ============================================================
# CHECK DATASET 2 IMAGE FORMAT
# ============================================================

for split in ["train", "valid", "test"]:

    image_dir = DATASET2_ROOT / split / "images"

    formats = {}

    for path in image_dir.iterdir():
        if path.is_file():
            ext = path.suffix.lower()
            formats[ext] = formats.get(ext, 0) + 1

    print(f"{split.upper()}:")
    print(formats)

TRAIN:
{'.jpg': 7056}
VALID:
{'.jpg': 882}
TEST:
{'.jpg': 872}


## 8. Create Final Combined Dataset

Create the final integrated dataset from BTXRD and Dataset 2.

All images will be stored as PNG files at 224 × 224 pixels.

The original datasets will not be modified.

Each image will receive a dataset prefix to prevent filename collisions.

A separate metadata.csv file will be created for each train, validation, and test split.

In [14]:
# ============================================================
# CREATE FINAL COMBINED DATASET
# ============================================================

import shutil
from PIL import Image

# Final combined dataset location
COMBINED_ROOT = PROCESSED_ROOT / "combined"

# Create clean split folders
for split in ["train", "valid", "test"]:

    image_dir = COMBINED_ROOT / split / "images"

    image_dir.mkdir(
        parents=True,
        exist_ok=True
    )

print("Final combined dataset location:")
print(COMBINED_ROOT)


# ============================================================
# FUNCTION: CONVERT AND SAVE IMAGE AS PNG
# ============================================================

def save_as_png(source_path, target_path):

    image = Image.open(source_path)

    # Convert to RGB for consistent PNG output
    if image.mode != "RGB":
        image = image.convert("RGB")

    image.save(
        target_path,
        format="PNG"
    )


# ============================================================
# PROCESS ONE SPLIT
# ============================================================

def process_split(split_name, btxrd_df, dataset2_df):

    destination = COMBINED_ROOT / split_name / "images"

    metadata_rows = []

    copied_btxrd = 0
    copied_dataset2 = 0
    failed = 0


    # --------------------------------------------------------
    # BTXRD
    # --------------------------------------------------------

    for _, row in btxrd_df.iterrows():

        source = BTXRD_IMAGES / (
            Path(row["filename"]).stem + ".png"
        )

        new_filename = "BTXRD_" + source.stem + ".png"

        target = destination / new_filename

        try:

            if source.exists():

                shutil.copy2(source, target)

                metadata_rows.append({
                    "filename": new_filename,
                    "cancer": int(row["cancer"]),
                    "dataset": "BTXRD",
                    "split": split_name
                })

                copied_btxrd += 1

            else:
                failed += 1
                print("BTXRD missing:", source)

        except Exception as e:

            failed += 1
            print("BTXRD failed:", source, "|", e)


    # --------------------------------------------------------
    # DATASET 2
    # --------------------------------------------------------

    source_dir = DATASET2_ROOT / split_name / "images"

    for _, row in dataset2_df.iterrows():

        source = source_dir / row["filename"]

        new_filename = (
            "Dataset2_" +
            Path(row["filename"]).stem +
            ".png"
        )

        target = destination / new_filename

        try:

            if source.exists():

                save_as_png(source, target)

                metadata_rows.append({
                    "filename": new_filename,
                    "cancer": int(row["cancer"]),
                    "dataset": "Dataset2",
                    "split": split_name
                })

                copied_dataset2 += 1

            else:
                failed += 1
                print("Dataset2 missing:", source)

        except Exception as e:

            failed += 1
            print("Dataset2 failed:", source, "|", e)


    # --------------------------------------------------------
    # SAVE METADATA
    # --------------------------------------------------------

    split_metadata = pd.DataFrame(metadata_rows)

    metadata_path = (
        COMBINED_ROOT /
        split_name /
        "metadata.csv"
    )

    split_metadata.to_csv(
        metadata_path,
        index=False
    )


    # --------------------------------------------------------
    # REPORT
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print(split_name.upper())
    print("=" * 60)

    print("BTXRD images:", copied_btxrd)
    print("Dataset 2 images:", copied_dataset2)
    print("Total images:", len(split_metadata))
    print("Failed:", failed)
    print("Metadata:", metadata_path)


# ============================================================
# PROCESS TRAIN
# ============================================================

process_split(
    "train",
    btxrd_train,
    dataset2_standard[
        dataset2_standard["split"] == "train"
    ]
)


# ============================================================
# PROCESS VALID
# ============================================================

process_split(
    "valid",
    btxrd_valid,
    dataset2_standard[
        dataset2_standard["split"] == "valid"
    ]
)


# ============================================================
# PROCESS TEST
# ============================================================

process_split(
    "test",
    btxrd_test,
    dataset2_standard[
        dataset2_standard["split"] == "test"
    ]
)


print("\n" + "=" * 60)
print("FINAL COMBINED DATASET CREATED")
print("=" * 60)
print(COMBINED_ROOT)

Final combined dataset location:
C:\Users\DELL\Downloads\mlproject\bone-cancer-detection\data\processed\combined

TRAIN
BTXRD images: 2996
Dataset 2 images: 7056
Total images: 10052
Failed: 0
Metadata: C:\Users\DELL\Downloads\mlproject\bone-cancer-detection\data\processed\combined\train\metadata.csv

VALID
BTXRD images: 375
Dataset 2 images: 882
Total images: 1257
Failed: 0
Metadata: C:\Users\DELL\Downloads\mlproject\bone-cancer-detection\data\processed\combined\valid\metadata.csv

TEST
BTXRD images: 375
Dataset 2 images: 872
Total images: 1247
Failed: 0
Metadata: C:\Users\DELL\Downloads\mlproject\bone-cancer-detection\data\processed\combined\test\metadata.csv

FINAL COMBINED DATASET CREATED
C:\Users\DELL\Downloads\mlproject\bone-cancer-detection\data\processed\combined


## 9. Verify Final Combined Dataset

Verify that every image in the combined train, validation, and test folders has a corresponding metadata entry.

Also verify that all final images are PNG and 224 × 224 pixels.

In [18]:
# ============================================================
# VERIFY FINAL COMBINED DATASET
# ============================================================

from PIL import Image

for split in ["train", "valid", "test"]:

    print("\n" + "=" * 60)
    print(split.upper())
    print("=" * 60)

    image_dir = COMBINED_ROOT / split / "images"
    metadata_path = COMBINED_ROOT / split / "metadata.csv"

    # Get image files
    image_files = [
        p for p in image_dir.iterdir()
        if p.is_file()
    ]

    # Load metadata
    df = pd.read_csv(metadata_path)

    print("Images in folder:", len(image_files))
    print("Rows in metadata:", len(df))

    # Check extensions
    extensions = {}

    for path in image_files:
        ext = path.suffix.lower()
        extensions[ext] = extensions.get(ext, 0) + 1

    print("Image formats:", extensions)

    # Check image dimensions
    sizes = set()

    for path in image_files[:100]:
        with Image.open(path) as img:
            sizes.add(img.size)

    print("Image sizes found in sample:", sizes)

    # Check whether every metadata filename exists
    missing = []

    for filename in df["filename"]:
        if not (image_dir / filename).exists():
            missing.append(filename)

    print("Metadata files missing from folder:", len(missing))

    # Check class distribution
    print("\nClass distribution:")
    print(df["cancer"].value_counts().sort_index())

print("\n" + "=" * 60)
print("VERIFICATION COMPLETE")
print("=" * 60)


TRAIN
Images in folder: 10052
Rows in metadata: 10052
Image formats: {'.png': 10052}
Image sizes found in sample: {(224, 224)}
Metadata files missing from folder: 0

Class distribution:
cancer
0    6697
1    3355
Name: count, dtype: int64

VALID
Images in folder: 1257
Rows in metadata: 1257
Image formats: {'.png': 1257}
Image sizes found in sample: {(224, 224)}
Metadata files missing from folder: 0

Class distribution:
cancer
0    825
1    432
Name: count, dtype: int64

TEST
Images in folder: 1247
Rows in metadata: 1247
Image formats: {'.png': 1247}
Image sizes found in sample: {(224, 224)}
Metadata files missing from folder: 0

Class distribution:
cancer
0    829
1    418
Name: count, dtype: int64

VERIFICATION COMPLETE
